# IBM Sales Opportunity Model

**Issue #4: Loading, cleaning, feature definitions, splitting and preprocessing.** No prediction model is trained at this checkpoint.

IBM was selected because its research experiments found more useful win/loss patterns than Maven. This is an academic IBM-data baseline, not proof that the model will work for every business.

We predict eventual **Won (1)** versus **Loss (0)** for one opportunity, using information intended to be available when scoring. A win probability will support sales decisions, not determine which deals must be pursued.

## 1. Imports and constants

Use the already-connected kernel. These imports support table operations, stratified splitting and standard preprocessing; no package installation is needed here. Model-training and SHAP imports will be added in later issues. The output should confirm that the setup cell ran.

In [10]:
from pathlib import Path  # Build file paths without machine-specific directories.

import numpy as np  # Numerical transformations and checks.
import pandas as pd  # CSV loading and table operations.
from IPython.display import display  # Show compact tables in the notebook.

from sklearn.model_selection import train_test_split  # Split rows while preserving class proportions.
from sklearn.compose import ColumnTransformer  # Transform different columns separately.
from sklearn.pipeline import Pipeline  # Apply preparation steps in order.
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_SEED = 42
id_column = "Opportunity Number"
target_column = "Opportunity Result"
amount_column = "Opportunity Amount USD"

# Support running from either the repository root or its notebooks directory.
repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
dataset_path = repository_root / "datasets" / "IBM-sales-opportunity.csv"

print("Issue #4 imports and constants are ready.")

Issue #4 imports and constants are ready.


## 2. Load the dataset

Read the local IBM CSV and inspect only its dimensions, columns and outcomes. `keep_default_na=False` preserves the real category `None`; `na_values=[""]` treats blank fields as missing. No data is downloaded.

Expect 78,025 raw rows and 19 columns. These counts include duplicates, which the next section removes.

In [11]:
raw_df = pd.read_csv(dataset_path, keep_default_na=False, na_values=[""])

print(f"Raw dataset: {len(raw_df):,} rows and {len(raw_df.columns)} columns.")
display(pd.DataFrame({"column": raw_df.columns}))

# Brackets select the outcome column; value_counts counts its labels.
display(raw_df[target_column].value_counts(dropna=False).rename("raw rows").to_frame())
raw_won_prevalence = raw_df[target_column].eq("Won").mean()
print(f"Raw Won prevalence: {raw_won_prevalence:.2%}")

Raw dataset: 78,025 rows and 19 columns.


,column
0,Opportunity Number
1,Supplies Subgroup
2,Supplies Group
3,Region
4,Route To Market
5,Elapsed Days In Sales Stage
6,Opportunity Result
7,Sales Stage Change Count
8,Total Days Identified Through Closing
9,Total Days Identified Through Qualified


,raw rows
Opportunity Result,
Loss,60398
Won,17627


Raw Won prevalence: 22.59%


## 3. Clean records and define the target

Remove exact duplicate rows so the same record is not counted twice.

Some opportunity numbers still have differing details after exact duplicates are removed. The CSV does not explain whether these are line items, updates or separate snapshots; treating them as independent opportunities could duplicate information or leak it across partitions. We therefore conservatively exclude **all** rows for those ambiguous IDs rather than guessing which record to keep.

Map `Won` to `1` and `Loss` to `0`. `.map(...)` translates labels; an unexpected label becomes missing and must fail the check, not become a loss. `assert` stops execution when a required condition is false.

Expect 55 exact duplicates removed, followed by 278 rows from 137 ambiguous IDs, leaving **77,692 unique opportunities**: 17,500 Won and 60,192 Loss. Each retained row represents one opportunity, so the study is limited to these single-record opportunities.

In [12]:
assert raw_df[id_column].notna().all(), "Every record needs an opportunity ID."
deduplicated_df = raw_df.drop_duplicates()

# keep=False marks every remaining row that shares its opportunity ID.
ambiguous_rows = deduplicated_df[id_column].duplicated(keep=False)
ambiguous_id_count = deduplicated_df.loc[ambiguous_rows, id_column].nunique()
# ~ reverses the mask; .loc keeps only unambiguous rows; .copy makes a new table.
cleaned_df = deduplicated_df.loc[~ambiguous_rows].copy()

cleaned_df["target"] = cleaned_df[target_column].map({"Won": 1, "Loss": 0})
assert cleaned_df["target"].notna().all(), "Unexpected or missing outcome label."
cleaned_df["target"] = cleaned_df["target"].astype(int)
assert cleaned_df[id_column].is_unique, "Each retained opportunity must be unique."
assert cleaned_df[amount_column].ge(0).all(), "Amount must be present and nonnegative."

cleaning_counts = pd.Series({
    "Raw rows": len(raw_df),
    "Exact duplicate rows removed": len(raw_df) - len(deduplicated_df),
    "Ambiguous IDs excluded": ambiguous_id_count,
    "Ambiguous rows removed": int(ambiguous_rows.sum()),
    "Retained unique opportunities": len(cleaned_df),
}, name="count")
display(cleaning_counts.to_frame())
display(cleaned_df[target_column].value_counts().rename("opportunities").to_frame())
print(f"Cleaned Won prevalence: {cleaned_df['target'].mean():.2%}")

,count
Raw rows,78025
Exact duplicate rows removed,55
Ambiguous IDs excluded,137
Ambiguous rows removed,278
Retained unique opportunities,77692


,opportunities
Opportunity Result,
Loss,60192
Won,17500


Cleaned Won prevalence: 22.52%


## 4. Define the candidate inputs

A **feature set** is a list of inputs supplied to a model. The four sets answer different research questions:

- **Reference 7:** establish the performance of the richer IBM-specific specification, including product subgroup and region.
- **Portable 5:** check whether product/geography can be removed while retaining similar performance.
- **Lean 3:** keep amount, prior-client revenue and route to check whether the two client-size fields add enough value to justify collecting them.
- **Limited-History 4:** check how useful prediction remains when prior-client revenue is unavailable, using the other four inputs.

Comparing these sets is practical feature selection: test the value of keeping or removing inputs without repeating the entire earlier audit. The table should show exactly which inputs belong to each specification.

History codes `0-4` and client-size codes `1-5` identify groups, not measured money or headcount. Code `4` is not mathematically twice code `2`, so treating them as categories avoids inventing numerical distances. Missing history is different from verified zero, and code `0` alone does not establish a new customer.

IDs, outcomes, later sales durations/ratios, competitor and redundant group/deal-size fields are excluded. Amount and history remain conditional on their availability before the outcome; this CSV cannot verify their timing.

In [13]:
# Each dictionary key names a model specification; its value lists the input columns.
feature_sets = {
    "Reference 7": [
        amount_column, "Revenue From Client Past Two Years", "Route To Market",
        "Client Size By Revenue", "Client Size By Employee Count",
        "Supplies Subgroup", "Region",
    ],
    "Portable 5": [
        amount_column, "Revenue From Client Past Two Years", "Route To Market",
        "Client Size By Revenue", "Client Size By Employee Count",
    ],
    "Lean 3": [
        amount_column, "Revenue From Client Past Two Years", "Route To Market",
    ],
    "Limited-History 4": [
        amount_column, "Route To Market",
        "Client Size By Revenue", "Client Size By Employee Count",
    ],
}

# Selecting a list of columns returns a DataFrame containing only those inputs.
missing_inputs = cleaned_df[feature_sets["Reference 7"]].isna().sum()
assert missing_inputs.eq(0).all(), "Unexpected missing inputs: review before preprocessing."

feature_membership = pd.DataFrame(index=feature_sets["Reference 7"])
for specification, feature_columns in feature_sets.items():
    assert not {id_column, target_column, "target"}.intersection(feature_columns)
    feature_membership[specification] = feature_membership.index.isin(feature_columns)
display(feature_membership.replace({True: "Yes", False: "-"}))

,Reference 7,Portable 5,Lean 3,Limited-History 4
Opportunity Amount USD,Yes,Yes,Yes,Yes
Revenue From Client Past Two Years,Yes,Yes,Yes,-
Route To Market,Yes,Yes,Yes,Yes
Client Size By Revenue,Yes,Yes,-,Yes
Client Size By Employee Count,Yes,Yes,-,Yes
Supplies Subgroup,Yes,-,-,-
Region,Yes,-,-,-


## 5. Separate training, calibration, validation and test

The approximate **50/10/20/20 split is a practical allocation for this relatively large dataset**, not a universal machine-learning rule. It leaves substantial, separate samples for each job:

- **Training (~50%, 38,845 records):** Only these records learn preprocessing parameters and the main predictive patterns. Nearly 39,000 opportunities provide a large training sample while leaving enough data for comparison, probability adjustment and final evaluation.
- **Calibration (~10%, 7,769 records):** Reserve these records to learn an optional adjustment to raw probabilities, not the main predictive patterns. A simple adjustment needs less data than classifier training, so this is a substantial sample for testing it without taking too much from training. Keep it separate from training and validation so the adjustment is learned on records the classifier did not train on and judged on different records.
- **Validation (~20%, 15,539 records):** Compare feature sets and algorithms, then choose calibration and the classification threshold. This large comparison sample helps reduce random variation without touching test results.
- **Test (~20%, 15,539 records):** Reserve these records until features, model, calibration and threshold are fixed. They estimate performance on records not used to fit or choose the approach in this notebook.

**What calibration checks:** among many comparable opportunities given about a 70% chance, do approximately 70% actually win? It checks whether probabilities are realistic across records, not whether one outcome is guaranteed.

**Why stratify?** There are substantially more Loss than Won opportunities. `stratify` keeps approximately the same Won/Loss proportion in each partition, avoiding accidental differences in the class mix caused by random splitting.

**Why fixed seeds?** `random_state` makes the partitions repeatable for the same input data, supporting reproducibility. Seed 42 and the subsequent seeds are simply fixed choices, not mathematically special values.

Cleaning leaves one unique row per opportunity, so `train_test_split` can separate rows directly. Reserve 20% for test, then 25% of the remaining 80% for validation, then one-sixth of the remaining 60% for calibration; integer rounding gives the counts above. The summary checks total row coverage and shows each group's size and win rate.

This changes membership from the earlier research split, but the IBM population has already been inspected. Later results are therefore an internal repeated evaluation, not a completely untouched or external test; absent dates/customer IDs also limit time-based and customer-independent claims. The extra development repeat belongs to Issue #6.

In [15]:
# stratify preserves roughly the same Won/Loss ratio; the seed makes the split repeatable.
development_df, test_df = train_test_split(
    cleaned_df, test_size=0.20, stratify=cleaned_df["target"], random_state=RANDOM_SEED,
)

# A quarter of the remaining 80% is about 20% of the original dataset.
fit_df, validation_df = train_test_split(
    development_df, test_size=0.25, stratify=development_df["target"],
    random_state=RANDOM_SEED + 1,
)

# One sixth of the remaining 60% is about 10%, leaving about 50% for training.
train_df, calibration_df = train_test_split(
    fit_df, test_size=1 / 6, stratify=fit_df["target"], random_state=RANDOM_SEED + 2,
)

In [16]:
partitions = {
    "Training": train_df,
    "Calibration": calibration_df,
    "Validation": validation_df,
    "Test": test_df,
}

# Cleaning already checked unique IDs; confirm that the splits retain every row.
assert sum(len(frame) for frame in partitions.values()) == len(cleaned_df)

partition_rows = []
for partition_name, frame in partitions.items():
    partition_rows.append({
        "partition": partition_name,
        "opportunities": len(frame),
        "share": len(frame) / len(cleaned_df),
        "Won prevalence": frame["target"].mean(),
    })
partition_summary = pd.DataFrame(partition_rows).set_index("partition")
display(partition_summary.round(4))
print("Partition totals match the cleaned dataset.")

,opportunities,share,Won prevalence
partition,,,
Training,38845,0.5,0.2253
Calibration,7769,0.1,0.2253
Validation,15539,0.2,0.2252
Test,15539,0.2,0.2252


Partition totals match the cleaned dataset.


## 6. Prepare model inputs

**Why one-hot encoding?** These algorithms require numerical inputs. One-hot encoding creates 0/1 indicator columns without pretending categories have mathematical distance; IBM history/size codes are group labels, not continuous measurements. `sparse_output=False` returns an ordinary dense numeric array.

**Why `log1p` amount?** Amounts range from small to very large, so compressing the range reduces the influence of extreme numerical sizes; `log(1 + amount)` also safely accepts zero. Earlier IBM research found raw and log amount gave similar predictive performance. We retain the log form as a clean, stable representation, not because larger deals are assumed more likely to win.

**Why standard scaling?** `StandardScaler` puts log amount on a standardized scale using the training mean and standard deviation. Logistic Regression is sensitive to feature scale; trees do not require scaling, but sharing the preprocessing structure keeps comparisons consistent.

**Why fit only on training?** Encoders learn category lists and scalers learn numerical parameters from training rows only. Letting calibration, validation or test records influence these transformations would leak evaluation information into model preparation.

**Why `handle_unknown="ignore"`?** A category absent from training receives all-zero indicators for its field, allowing preprocessing to continue without an encoding failure. This does **not** mean the model understands the new category; the application should later flag it as an input/applicability limitation. That reliability flag is not implemented here.

**Why no imputation?** The audited IBM candidate fields are complete, so no replacement values need to be invented. Unknown prior-client history must not be silently converted to zero: missing information and verified no previous revenue are different business situations, and zero category indicators do not change that distinction.

`ColumnTransformer` applies the appropriate treatment to each column, `Pipeline` keeps log transformation before scaling, and `remainder="drop"` excludes unselected columns. The helper returns a **new, unfitted** preprocessor for each feature list; fit it on training rows and inspect a small transformed batch. Expect four finite numeric batches without fitting on evaluation data.

In [17]:
def build_preprocessor(feature_columns):
    """Return an unfitted ColumnTransformer for a list of approved feature names.

    Amount is log-transformed and scaled. All other selected inputs are
    one-hot encoded as categories; no classifier is fitted here.
    """
    amount_features = [amount_column]
    categorical_features = [column for column in feature_columns if column != amount_column]

    amount_transformer = Pipeline([
        # one-to-one preserves the original column name after log transformation.
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",  # An unseen category gets zeros for this field's indicators.
        sparse_output=False,      # Produce an ordinary numeric array.
    )

    # The output order is encoded categories first, followed by the amount column.
    return ColumnTransformer([
        ("categorical", categorical_encoder, categorical_features),
        ("amount", amount_transformer, amount_features),
    ], remainder="drop")

In [18]:
preprocessors = {}
preprocessing_rows = []

for specification, feature_columns in feature_sets.items():
    training_inputs = train_df[feature_columns]  # Select only this candidate's inputs.
    preprocessor = build_preprocessor(feature_columns)

    # fit learns categories and amount scaling from training rows only.
    preprocessor.fit(training_inputs)
    # transform applies those learned rules to a small batch without refitting.
    training_batch = preprocessor.transform(training_inputs.head(8))
    assert np.isfinite(training_batch).all(), "Preprocessing produced missing or infinite values."

    preprocessors[specification] = preprocessor
    preprocessing_rows.append({
        "specification": specification,
        "original inputs": len(feature_columns),
        "training rows fitted": len(training_inputs),
        "batch rows": training_batch.shape[0],
        "encoded columns": training_batch.shape[1],
    })

preprocessing_summary = pd.DataFrame(preprocessing_rows).set_index("specification")
display(preprocessing_summary)
print("All four training batches transformed successfully; no classifier has been trained.")

,original inputs,training rows fitted,batch rows,encoded columns
specification,,,,
Reference 7,7,38845,8,39
Portable 5,5,38845,8,21
Lean 3,3,38845,8,11
Limited-History 4,4,38845,8,16


All four training batches transformed successfully; no classifier has been trained.


## Issue #4 checkpoint

**Simplified and rerun from the beginning in the connected kernel.**

| Requirement | Result |
|---|---|
| Established cleaning and target | 77,692 unique opportunities; 17,500 Won / 60,192 Loss. Core cleaning assertions retained. |
| Four feature specifications | Reference 7, Portable 5, Lean 3 and Limited-History 4 defined with complete inputs. |
| Sequential stratified splits | 38,845 training / 7,769 calibration / 15,539 validation / 15,539 test; total row coverage checked. Unique IDs were checked during cleaning. |
| Training-only preprocessing | Fitted on 38,845 training rows. All four 8-row batches passed finite-value checks; 39 / 21 / 11 / 16 encoded columns respectively. |
| Unseen categories | `handle_unknown="ignore"` avoids encoding failure; an unseen value gets zero indicators for its field, not learned meaning. Later application reliability must flag it. |
| Readability | Short Markdown explanations, useful syntax comments, direct split calls and one documented `build_preprocessor` helper. |

The grouped-split helper, pairwise overlap loop and unseen-route rejection test have been removed. This update supersedes those choices in the earlier plan. No classifier, calibration adjustment or final test score has been produced.

**Issue #4 establishes the reproducible input pipeline only.** Model training begins in Issue #5 so preprocessing can be reviewed independently before it influences model comparisons.

Issue #4 is ready for review and closure. Suggested commit: `Build IBM preprocessing and feature pipeline`. No commit or GitHub issue action is performed here.

**Next: Issue #5**, the prevalence baseline and Logistic Regression / Random Forest / XGBoost validation comparison. That work has not started. Use fresh model pipelines with `build_preprocessor`, and do not fit transformations on evaluation data.